<a href="https://colab.research.google.com/github/oguzhanguler1/titanic_competition/blob/main/submission2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [218]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [219]:
dataset = pd.read_csv("train.csv")
test_dataset = pd.read_csv("test.csv")

In [220]:
test_passenger_ids = test_dataset["PassengerId"]

In [221]:
dataset.isnull().sum().sort_values(ascending=False)

,0
Cabin,687
Age,177
Embarked,2
PassengerId,0
Name,0
Pclass,0
Survived,0
Sex,0
Parch,0
SibSp,0


In [222]:
test_dataset.isnull().sum().sort_values(ascending=False)

,0
Cabin,327
Age,86
Fare,1
Name,0
Pclass,0
PassengerId,0
Sex,0
Parch,0
SibSp,0
Ticket,0


In [223]:
dataset["Sex"] = dataset["Sex"].map({
    "male": 0,
    "female": 1
})

test_dataset["Sex"] = test_dataset["Sex"].map({
    "male": 0,
    "female": 1
})

In [224]:
dataset["Title"] = dataset["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)
test_dataset["Title"] = test_dataset["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)

<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_4088/624550131.py:1: SyntaxWarning: invalid escape sequence '\.'
  dataset["Title"] = dataset["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)
/tmp/ipykernel_4088/624550131.py:2: SyntaxWarning: invalid escape sequence '\.'
  test_dataset["Title"] = test_dataset["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)


In [225]:
rare_titles = [
    "Lady", "Countess", "Capt", "Col", "Don", "Dr",
    "Major", "Rev", "Sir", "Jonkheer", "Dona"
]

dataset["Title"] = dataset["Title"].replace(rare_titles, "Rare")
test_dataset["Title"] = test_dataset["Title"].replace(rare_titles, "Rare")

dataset["Title"] = dataset["Title"].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
})

test_dataset["Title"] = test_dataset["Title"].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
})

In [226]:
dataset["Age"] = dataset.groupby(["Title", "Pclass"])["Age"].transform(
    lambda x: x.fillna(x.median())
)

dataset["Age"] = dataset["Age"].fillna(dataset["Age"].median())

In [227]:
test_dataset["Age"] = test_dataset.groupby(["Title", "Pclass"])["Age"].transform(
    lambda x: x.fillna(x.median())
)

test_dataset["Age"] = test_dataset["Age"].fillna(dataset["Age"].median())

In [228]:
dataset["AgeBin"] = pd.cut(
    dataset["Age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=[0, 1, 2, 3, 4],
    include_lowest=True
).astype(int)

test_dataset["AgeBin"] = pd.cut(
    test_dataset["Age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=[0, 1, 2, 3, 4],
    include_lowest=True
).astype(int)

In [229]:
dataset["FamilySize"] = dataset["SibSp"] + dataset["Parch"] + 1
test_dataset["FamilySize"] = test_dataset["SibSp"] + test_dataset["Parch"] + 1

In [230]:
def family_group(size):
    if size == 1:
        return 0   # Alone
    elif size <= 4:
        return 1   # Small family
    else:
        return 2   # Large family

dataset["FamilyGroup"] = dataset["FamilySize"].apply(family_group)
test_dataset["FamilyGroup"] = test_dataset["FamilySize"].apply(family_group)

In [231]:
dataset["HasCabin"] = dataset["Cabin"].notnull().astype(int)
test_dataset["HasCabin"] = test_dataset["Cabin"].notnull().astype(int)

In [232]:
test_dataset["Fare"] = test_dataset["Fare"].fillna(dataset["Fare"].median())

In [233]:
dataset["FareBin"], fare_bins = pd.qcut(
    dataset["Fare"],
    4,
    labels=[0, 1, 2, 3],
    retbins=True,
    duplicates="drop"
)

dataset["FareBin"] = dataset["FareBin"].astype(int)

In [234]:
test_dataset["FareBin"] = pd.cut(
    test_dataset["Fare"],
    bins=fare_bins,
    labels=[0, 1, 2, 3],
    include_lowest=True
)

In [235]:
from sklearn.preprocessing import OneHotEncoder
title_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)
title_encoded_train = title_encoder.fit_transform(dataset[["Title"]])
title_encoded_test = title_encoder.transform(test_dataset[["Title"]])

In [236]:
title_columns = title_encoder.get_feature_names_out(["Title"])
title_columns

array(['Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs',
       'Title_Rare'], dtype=object)

In [237]:
title_train_df = pd.DataFrame(
    title_encoded_train,
    columns=title_columns,
    index=dataset.index
)

title_test_df = pd.DataFrame(
    title_encoded_test,
    columns=title_columns,
    index=test_dataset.index
)

In [238]:
base_features = [
    "Pclass",
    "Sex",
    "AgeBin",
    "FamilySize",
    "FamilyGroup",
    "HasCabin",
    "FareBin"
]

In [239]:
X_base = dataset[base_features]
X_test_base = test_dataset[base_features]

In [240]:
X = pd.concat(
    [X_base, title_train_df],
    axis=1
)

X_test = pd.concat(
    [X_test_base, title_test_df],
    axis=1
)

In [241]:
y = dataset["Survived"]

In [242]:
X.head()

,Pclass,Sex,AgeBin,FamilySize,FamilyGroup,HasCabin,FareBin,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,3,0,2,2,1,0,0,0.0,0.0,1.0,0.0,0.0
1,1,1,3,2,1,1,3,0.0,0.0,0.0,1.0,0.0
2,3,1,2,1,0,0,1,0.0,1.0,0.0,0.0,0.0
3,1,1,2,2,1,1,3,0.0,0.0,0.0,1.0,0.0
4,3,0,2,1,0,0,1,0.0,0.0,1.0,0.0,0.0


In [243]:
X_test.head()

,Pclass,Sex,AgeBin,FamilySize,FamilyGroup,HasCabin,FareBin,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,3,0,2,1,0,0,0,0.0,0.0,1.0,0.0,0.0
1,3,1,3,2,1,0,0,0.0,0.0,0.0,1.0,0.0
2,2,0,4,1,0,0,1,0.0,0.0,1.0,0.0,0.0
3,3,0,2,1,0,0,1,0.0,0.0,1.0,0.0,0.0
4,3,1,2,3,1,0,1,0.0,0.0,0.0,1.0,0.0


In [244]:
print(X.columns)
print(X_test.columns)

Index(['Pclass', 'Sex', 'AgeBin', 'FamilySize', 'FamilyGroup', 'HasCabin',
       'FareBin', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs',
       'Title_Rare'],
      dtype='object')
Index(['Pclass', 'Sex', 'AgeBin', 'FamilySize', 'FamilyGroup', 'HasCabin',
       'FareBin', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs',
       'Title_Rare'],
      dtype='object')


In [245]:
X.isnull().sum().sort_values(ascending=False)

,0
Pclass,0
Sex,0
AgeBin,0
FamilySize,0
FamilyGroup,0
HasCabin,0
FareBin,0
Title_Master,0
Title_Miss,0
Title_Mr,0


In [246]:
X_test.isnull().sum().sort_values(ascending=False)

,0
Pclass,0
Sex,0
AgeBin,0
FamilySize,0
FamilyGroup,0
HasCabin,0
FareBin,0
Title_Master,0
Title_Miss,0
Title_Mr,0


In [247]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.1,
    random_state=43,
    stratify=y
)

In [248]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

svc_model = SVC()

svc_model.fit(X_train, y_train)

svc_pred = svc_model.predict(X_val)

print("SVC Accuracy:", accuracy_score(y_val, svc_pred))
print(classification_report(y_val, svc_pred))
print(confusion_matrix(y_val, svc_pred))

SVC Accuracy: 0.8111111111111111
              precision    recall  f1-score   support

           0       0.84      0.85      0.85        55
           1       0.76      0.74      0.75        35

    accuracy                           0.81        90
   macro avg       0.80      0.80      0.80        90
weighted avg       0.81      0.81      0.81        90

[[47  8]
 [ 9 26]]


In [249]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    SVC(),
    X,
    y,
    cv=skf,
    scoring="accuracy"
)

print("CV Scores:", cv_scores)
print("Mean Accuracy:", cv_scores.mean())
print("Standard Deviation:", cv_scores.std())

CV Scores: [0.83798883 0.82022472 0.8258427  0.83146067 0.84831461]
Mean Accuracy: 0.8327663046889711
Standard Deviation: 0.009756606200828069


In [250]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),

    "Random Forest": RandomForestClassifier(
        random_state=42,
        n_estimators=200
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    ),

    "Hist Gradient Boosting": HistGradientBoostingClassifier(
        random_state=42
    ),

    "SVC": SVC(),

    "Scaled SVC": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC())
    ]),

    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            hidden_layer_sizes=(32, 16),
            activation="relu",
            max_iter=1000,
            random_state=42
        ))
    ])
}

In [251]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, pred)
    })

results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)
results_df

,Model,Accuracy
6,MLP,0.822222
5,Scaled SVC,0.822222
4,SVC,0.811111
3,Hist Gradient Boosting,0.811111
0,Logistic Regression,0.800000
2,Gradient Boosting,0.800000
1,Random Forest,0.788889


In [252]:
cv_results = []

for name, model in models.items():
    scores = cross_val_score(
        model,
        X,
        y,
        cv=skf,
        scoring="accuracy"
    )

    cv_results.append({
        "Model": name,
        "Mean Accuracy": scores.mean(),
        "Std": scores.std()
    })

cv_results_df = pd.DataFrame(cv_results).sort_values(by="Mean Accuracy", ascending=False)
cv_results_df

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


,Model,Mean Accuracy,Std
4,SVC,0.832766,0.009757
0,Logistic Regression,0.830507,0.014164
5,Scaled SVC,0.829396,0.005985
2,Gradient Boosting,0.829389,0.011776
3,Hist Gradient Boosting,0.823790,0.019358
6,MLP,0.818172,0.020054
1,Random Forest,0.817030,0.021416


In [253]:
final_model = SVC()

final_model.fit(X, y)

test_predictions = final_model.predict(X_test)

In [254]:
submission = pd.DataFrame({
    "PassengerId": test_passenger_ids,
    "Survived": test_predictions
})

submission.to_csv("submission2.csv", index=False)

submission.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


In [256]:
old_submission = pd.read_csv("submission.csv")
new_submission = pd.read_csv("submission2.csv")

(old_submission["Survived"] == new_submission["Survived"]).mean()

np.float64(0.9832535885167464)

In [257]:
# 1. Kaç tahmin değişmiş?
changed_mask = old_submission["Survived"] != new_submission["Survived"]
changed_mask.sum()

np.int64(7)

In [258]:
# 2. Değişen tahminleri gör
changed = pd.DataFrame({
    "PassengerId": old_submission.loc[changed_mask, "PassengerId"],
    "OldPrediction": old_submission.loc[changed_mask, "Survived"],
    "NewPrediction": new_submission.loc[changed_mask, "Survived"]
})

changed

,PassengerId,OldPrediction,NewPrediction
32,924,0,1
33,925,0,1
244,1136,0,1
339,1231,0,1
344,1236,0,1
392,1284,0,1
417,1309,0,1


In [259]:
# 3. Değişen yolcuların özelliklerine bak
test_dataset[test_dataset["PassengerId"].isin(changed["PassengerId"])][[
    "PassengerId",
    "Pclass",
    "Sex",
    "Title",
    "Age",
    "AgeBin",
    "FamilySize",
    "FamilyGroup",
    "HasCabin",
    "Fare",
    "FareBin"
]]

,PassengerId,Pclass,Sex,Title,Age,AgeBin,FamilySize,FamilyGroup,HasCabin,Fare,FareBin
32,924,3,1,Mrs,33.0,2,4,1,0,20.5750,2
33,925,3,1,Mrs,28.0,2,4,1,0,23.4500,2
244,1136,3,0,Master,7.0,0,4,1,0,23.4500,2
339,1231,3,0,Master,7.0,0,1,0,0,7.2292,0
344,1236,3,0,Master,7.0,0,3,1,0,14.5000,2
392,1284,3,0,Master,13.0,1,3,1,0,20.2500,2
417,1309,3,0,Master,7.0,0,3,1,0,22.3583,2
